In [50]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [75]:
import mlflow
import mlflow.sklearn

MLFLOW_TRACKING_URI = (
    "sqlite:////Users/jinchoi/Documents/Python/mlops/mlflow/db/mlflow.db"
)
EXPERIMENT_NAME = "zoomcamp-model-v2"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print(
    "Active experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
)

2026/03/09 22:13:33 INFO mlflow.tracking.fluent: Experiment with name 'zoomcamp-model-v2' does not exist. Creating a new experiment.
2026/03/09 22:13:33 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('zoomcamp-model-v2', 'default', None, 'active', 1773112413379, 1773112413379)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "/Users/jinchoi/Documents/Python/.venv/lib/python3.13/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/jinchoi/Docume

MlflowException: (sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('zoomcamp-model-v2', 'default', None, 'active', 1773112413379, 1773112413379)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [3]:
# Load Jan-Nov 2024 green taxi data
YEAR = 2024
TRAIN_MONTHS = list(range(1, 11))
TEST_MONTH = 11

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet"


def load_month(year: int, month: int) -> pd.DataFrame:
    url = BASE_URL.format(year=year, month=month)
    print(f"  Downloading {year}-{month:02d} ...")
    return pd.read_parquet(url)


def load_months(year: int, months: list) -> pd.DataFrame:
    dfs = [load_month(year, month) for month in months]
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Total: {len(df):,} records\n")
    return df

In [4]:
df_train_raw = load_months(YEAR, TRAIN_MONTHS)
df_test_raw = load_month(YEAR, TEST_MONTH)

print(f"Train raw shape: {df_train_raw.shape}")
print(f"Test raw shape: {df_test_raw.shape}")

  Total: 554,002 records

Train raw shape: (554002, 20)
Test raw shape: (52222, 20)


In [5]:
# Feature setup + split once
FEATURE_COLS = [
    "VendorID",
    "passenger_count",
    "trip_distance_log",
    "RatecodeID",
    "trip_type",
    "store_and_fwd_flag",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend",
    "rush_hour",
    "trip_duration_min_log",
    "route",
]

TARGET = "fare_amount"

OHE_COLS = ["VendorID", "RatecodeID", "trip_type", "store_and_fwd_flag"]
TE_COLS = ["route"]
NUM_COLS = [
    "passenger_count",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend",
    "rush_hour",
    "trip_duration_min_log",
    "trip_distance_log",
]

In [6]:
def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Basic cleaning and feature engineering
    target = "fare_amount"

    df["lpep_pickup_datetime"] = pd.to_datetime(df["lpep_pickup_datetime"])
    df["lpep_dropoff_datetime"] = pd.to_datetime(df["lpep_dropoff_datetime"])

    df["trip_duration_min"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["lpep_pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["lpep_pickup_datetime"].dt.month
    df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)
    df["rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

    # High-cardinality route feature
    df["route"] = (
        df["PULocationID"].astype("Int64").astype(str)
        + "_"
        + df["DOLocationID"].astype("Int64").astype(str)
    )

    mask = (
        df["trip_duration_min"].between(1, 180)
        & df["fare_amount"].between(0.5, 500)
        & df["trip_distance"].between(0.01, 100)
        & df["passenger_count"].between(0, 8)
    )

    df = df.loc[mask].copy()

    df["trip_distance_log"] = np.log1p(df["trip_distance"].copy())
    df["trip_duration_min_log"] = np.log1p(df["trip_duration_min"].copy())

    print("Rows after subsetting:", f"{len(df):,}")

    X = df[FEATURE_COLS].copy()
    y = np.log1p(df[target].copy())

    for col in OHE_COLS:
        X[col] = X[col].astype(str)

    return X, y

In [7]:
X_train_full, y_train_full = clean_and_engineer(df_train_raw)
X_test, y_test = clean_and_engineer(df_test_raw)

Rows after subsetting: 495,616
Rows after subsetting: 47,042


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.3, random_state=42
)

print("Train rows:", f"{len(X_train):,}")
print("Val rows  :", f"{len(X_val):,}")
print("Test rows :", f"{len(X_test):,}")

Train rows: 346,931
Val rows  : 148,685
Test rows : 47,042


In [9]:
def compute_vif_statsmodels(df_num: pd.DataFrame) -> pd.DataFrame:
    X = df_num.to_numpy()
    return pd.DataFrame(
        {
            "feature": df_num.columns,
            "vif": [variance_inflation_factor(X, i) for i in range(X.shape[1])],
        }
    ).sort_values("vif", ascending=False)


def vif_prune(df_num: pd.DataFrame, threshold: float = 5.0):
    kept = list(df_num.columns)
    dropped = []

    while len(kept) > 1:
        vif_df = compute_vif_statsmodels(df_num[kept])
        max_row = vif_df.iloc[0]
        max_vif = max_row["vif"]
        max_feature = max_row["feature"]
        if max_vif <= threshold:
            break
        dropped.append({"feature": max_feature, "vif": float(max_vif)})
        kept.remove(max_feature)

    final_vif = compute_vif_statsmodels(df_num[kept])
    return kept, dropped, final_vif

In [10]:
# Train-only numeric imputation + VIF pruning on numeric features
num_imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[NUM_COLS]),
    columns=NUM_COLS,
    index=X_train.index,
)

kept_num_cols, vif_dropped, final_vif = vif_prune(X_train_num, threshold=10.0)
print("Kept numeric columns:", kept_num_cols)
print("Dropped by VIF:", vif_dropped)
print(final_vif)

Kept numeric columns: ['passenger_count', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'is_weekend', 'rush_hour', 'trip_distance_log']
Dropped by VIF: [{'feature': 'trip_duration_min_log', 'vif': 33.070342445522805}]
             feature       vif
2   pickup_dayofweek  6.856550
1        pickup_hour  5.703569
6  trip_distance_log  4.670918
3       pickup_month  4.020050
4         is_weekend  3.094040
0    passenger_count  2.579909
5          rush_hour  1.728088


In [11]:
# Preprocessor factory with post-VIF numeric columns
def make_preprocessor(kept_num_cols):
    num_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    ohe_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    te_pipe = Pipeline(
        [
            (
                "te",
                TargetEncoder(
                    cols=TE_COLS,
                    smoothing=20,
                    handle_missing="value",
                    handle_unknown="value",
                ),
            ),
        ]
    )

    return ColumnTransformer(
        [
            ("num", num_pipe, kept_num_cols),
            ("ohe", ohe_pipe, OHE_COLS),
            ("te", te_pipe, TE_COLS),
        ]
    )


def rmse_from_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [12]:
# Baseline metric (no VIF/LASSO): Ridge on all numeric cols + OHE + TE
baseline_preprocessor = make_preprocessor(NUM_COLS)
baseline_model = Pipeline(
    [
        ("preprocess", baseline_preprocessor),
        ("reg", Ridge(alpha=1.0, random_state=42)),
    ]
)
baseline_model.fit(X_train, y_train)

baseline_val_pred_log = baseline_model.predict(X_val)
baseline_val_rmse = rmse_from_log(y_val, baseline_val_pred_log)
print(f"Baseline val RMSE (no VIF/LASSO): {baseline_val_rmse:.4f}")

Baseline val RMSE (no VIF/LASSO): 5.1992


In [13]:
# LASSO selection on post-VIF feature space (train only), tune alpha on validation
preprocessor_vif = make_preprocessor(kept_num_cols)
alphas = np.logspace(-4, 0, 12)

best_alpha = None
best_lasso_model = None
best_lasso_val_rmse = float("inf")
min_selected_features = 2

fallback_best_alpha = None
fallback_best_model = None
fallback_best_rmse = float("inf")

with mlflow.start_run(run_name="lasso_hyperparameter_tuning") as lasso_parent_run:
    mlflow.log_param("stage", "lasso_hpo")
    mlflow.log_param("min_selected_features", min_selected_features)
    mlflow.log_param("alpha_grid", ",".join([f"{a:.8f}" for a in alphas]))

    for alpha in alphas:
        with mlflow.start_run(run_name=f"lasso_alpha_{alpha:.6f}", nested=True):
            lasso_model = Pipeline(
                [
                    ("preprocess", preprocessor_vif),
                    ("reg", Lasso(alpha=float(alpha), max_iter=20000, random_state=42)),
                ]
            )
            lasso_model.fit(X_train, y_train)

            val_pred_log = lasso_model.predict(X_val)
            val_rmse = rmse_from_log(y_val, val_pred_log)
            n_selected = int(
                (np.abs(lasso_model.named_steps["reg"].coef_) > 1e-8).sum()
            )

            mlflow.log_param("alpha", float(alpha))
            mlflow.log_metric("val_rmse", val_rmse)
            mlflow.log_metric("n_selected_features", n_selected)

            print(f"alpha={alpha:.6f}  val_rmse={val_rmse:.4f}  selected={n_selected}")

            if val_rmse < fallback_best_rmse:
                fallback_best_rmse = val_rmse
                fallback_best_alpha = float(alpha)
                fallback_best_model = lasso_model

            if n_selected >= min_selected_features and val_rmse < best_lasso_val_rmse:
                best_lasso_val_rmse = val_rmse
                best_alpha = float(alpha)
                best_lasso_model = lasso_model

    if best_lasso_model is None:
        print(
            f"No alpha met min_selected_features={min_selected_features}; using RMSE-best fallback."
        )
        best_alpha = fallback_best_alpha
        best_lasso_model = fallback_best_model
        best_lasso_val_rmse = fallback_best_rmse

    mlflow.log_metric("best_lasso_val_rmse", best_lasso_val_rmse)
    mlflow.log_param("best_lasso_alpha", best_alpha)

print(f"Best LASSO alpha: {best_alpha:.6f}")
print(f"Post-VIF + LASSO val RMSE: {best_lasso_val_rmse:.4f}")

alpha=0.000100  val_rmse=5.9972  selected=13
alpha=0.000231  val_rmse=6.0466  selected=13
alpha=0.000534  val_rmse=6.0335  selected=12
alpha=0.001233  val_rmse=6.0009  selected=12
alpha=0.002848  val_rmse=5.9640  selected=7
alpha=0.006579  val_rmse=6.0139  selected=6
alpha=0.015199  val_rmse=6.3602  selected=1
alpha=0.035112  val_rmse=6.5420  selected=1
alpha=0.081113  val_rmse=7.2732  selected=1
alpha=0.187382  val_rmse=9.5029  selected=1
alpha=0.432876  val_rmse=13.8303  selected=1
alpha=1.000000  val_rmse=14.8832  selected=0
Best LASSO alpha: 0.002848
Post-VIF + LASSO val RMSE: 5.9640


In [14]:
# Extract selected features from best LASSO and run consistency checks
feature_names = best_lasso_model.named_steps["preprocess"].get_feature_names_out()
coefs = best_lasso_model.named_steps["reg"].coef_
coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})

selected_mask = np.abs(coefs) > 1e-8
selected_feature_names = feature_names[selected_mask].tolist()

# Verify transform compatibility on val/test and stable feature order
X_train_tx = best_lasso_model[:-1].transform(X_train)
X_val_tx = best_lasso_model[:-1].transform(X_val)

coef_df.loc[selected_mask, "abs_coef"] = coef_df["coef"].abs()

print("Total transformed features:", len(feature_names))
print("Selected non-zero features:", len(selected_feature_names))
print("Top selected features:")
print(coef_df.loc[selected_mask])

Total transformed features: 22
Selected non-zero features: 7
Top selected features:
                   feature      coef  abs_coef
3        num__pickup_month  0.003830  0.003830
4          num__is_weekend -0.006041  0.006041
5           num__rush_hour  0.002402  0.002402
6   num__trip_distance_log  0.452410  0.452410
9      ohe__RatecodeID_1.0 -0.061811  0.061811
13     ohe__RatecodeID_5.0  0.231092  0.231092
21               te__route  0.141696  0.141696


In [15]:
# Final model: tune Ridge on selected features, then refit on full train months and evaluate on next month test
selected_indices = [
    int(np.where(feature_names == name)[0][0]) for name in selected_feature_names
]

X_train_sel = X_train_tx[:, selected_indices]
X_val_sel = X_val_tx[:, selected_indices]

ridge_alphas = np.logspace(-4, 4, 50)
best_ridge_alpha = None
best_ridge_val_rmse = float("inf")

with mlflow.start_run(run_name="ridge_training_and_final_model") as ridge_run:
    mlflow.log_param("stage", "ridge_hpo_final")
    mlflow.log_param("selected_feature_count", len(selected_feature_names))
    mlflow.log_param("ridge_alpha_grid_size", len(ridge_alphas))

    for alpha in ridge_alphas:
        with mlflow.start_run(run_name=f"ridge_alpha_{alpha:.6f}", nested=True):
            ridge = Ridge(alpha=alpha, random_state=42)
            ridge.fit(X_train_sel, y_train)
            val_pred_log = ridge.predict(X_val_sel)
            val_rmse = rmse_from_log(y_val, val_pred_log)

            mlflow.log_param("alpha", float(alpha))
            mlflow.log_metric("val_rmse", val_rmse)

            print(f"Ridge alpha={alpha:.2f}  val_rmse={val_rmse:.4f}")
            if val_rmse < best_ridge_val_rmse:
                best_ridge_val_rmse = val_rmse
                best_ridge_alpha = alpha

    print(f"Best Ridge alpha (selected vars): {best_ridge_alpha}")
    print(f"Final Ridge-on-selected val RMSE: {best_ridge_val_rmse:.4f}")

    # Refit once on all prior-month training data
    final_preprocessor = make_preprocessor(kept_num_cols)
    X_train_full_tx = final_preprocessor.fit_transform(X_train_full, y_train_full)
    final_feature_names = final_preprocessor.get_feature_names_out()

    name_to_idx = {name: i for i, name in enumerate(final_feature_names)}
    final_selected_indices = [name_to_idx[name] for name in selected_feature_names]

    X_train_full_sel = X_train_full_tx[:, final_selected_indices]
    final_ridge = Ridge(alpha=best_ridge_alpha, random_state=42)
    final_ridge.fit(X_train_full_sel, y_train_full)

    # Evaluate once on test month
    X_test_tx_final = final_preprocessor.transform(X_test)
    X_test_sel_final = X_test_tx_final[:, final_selected_indices]
    test_pred_log = final_ridge.predict(X_test_sel_final)
    final_test_rmse = rmse_from_log(y_test, test_pred_log)
    print(f"Final test RMSE (Ridge on selected vars): {final_test_rmse:.4f}")

    mlflow.log_param("best_ridge_alpha", float(best_ridge_alpha))
    mlflow.log_metric("best_ridge_val_rmse", best_ridge_val_rmse)
    mlflow.log_metric("final_test_rmse", final_test_rmse)
    mlflow.log_param("selected_feature_count", len(selected_feature_names))

    mlflow.log_dict(
        {"selected_feature_names": selected_feature_names}, "selected_features.json"
    )
    mlflow.sklearn.log_model(final_ridge, name="final_model")
    logged_model_uri = f"runs:/{ridge_run.info.run_id}/final_model"

print("Logged model URI:", logged_model_uri)

Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.00  val_rmse=6.0905
Ridge alpha=0.01  val_rmse=6.0905
Ridge alpha=0.01  val_rmse=6.0905
Ridge alpha=0.01  val_rmse=6.0905
Ridge alpha=0.02  val_rmse=6.0905
Ridge alpha=0.03  val_rmse=6.0905
Ridge alpha=0.04  val_rmse=6.0905
Ridge alpha=0.06  val_rmse=6.0905
Ridge alpha=0.09  val_rmse=6.0905
Ridge alpha=0.13  val_rmse=6.0905
Ridge alpha=0.18  val_rmse=6.0905
Ridge alpha=0.27  val_rmse=6.0904
Ridge alpha=0.39  val_rmse=6.0904
Ridge alpha=0.57  val_rmse=6.0904
Ridge alpha=0.83  val_rmse=6.0904
Ridge alpha=1.21  val_rmse=6.0904
Ridge alpha=1.76  val_rmse=6.0903
Ridge alpha=2.56  val_rmse=6.0903
Ridge alpha=3.73  val_rmse=6.0902
Ridge alpha=5.

2026/03/09 17:02:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final test RMSE (Ridge on selected vars): 6.6117
Logged model URI: runs:/7b07ba1bcc744a12bd029d9fc68614dd/final_model


In [16]:
coef_df = pd.DataFrame(
    {"feature": selected_feature_names, "coef": final_ridge.coef_}
).sort_values("coef", key=lambda s: s.abs(), ascending=False)
coef_df.head(30)

,feature,coef
3,num__trip_distance_log,0.435478
6,te__route,0.171451
5,ohe__RatecodeID_5.0,0.166899
4,ohe__RatecodeID_1.0,-0.144632
1,num__is_weekend,-0.008803
0,num__pickup_month,0.006502
2,num__rush_hour,0.005005


In [17]:
# Inference-time: load model from MLflow and evaluate
loaded_model = mlflow.sklearn.load_model(logged_model_uri)
X_test_tx_loaded = final_preprocessor.transform(X_test)
X_test_sel_loaded = X_test_tx_loaded[:, final_selected_indices]
loaded_test_pred_log = loaded_model.predict(X_test_sel_loaded)
loaded_test_rmse = rmse_from_log(y_test, loaded_test_pred_log)

print("Loaded model URI:", logged_model_uri)
print(f"Loaded-model test RMSE: {loaded_test_rmse:.4f}")

Loaded model URI: runs:/7b07ba1bcc744a12bd029d9fc68614dd/final_model
Loaded-model test RMSE: 6.6117


In [18]:
# Optional: register the latest logged model
registered_model_name = "nyc-taxi-ridge"
model_details = mlflow.register_model(
    model_uri=logged_model_uri, name=registered_model_name
)
print("Registered model version:", model_details.version)

Registered model 'nyc-taxi-ridge' already exists. Creating a new version of this model...
2026/03/09 17:02:09 WARNING mlflow.tracking._model_registry.fluent: Run with id 7b07ba1bcc744a12bd029d9fc68614dd has no artifacts at artifact path 'final_model', registering model based on models:/m-048ef3a2e04c4295b7ac2478f848d8a5 instead


Registered model version: 8


Created version '8' of model 'nyc-taxi-ridge'.
